# Figure 1 synthetic NK diffusion-scale validation

Retains only the dense NK and Laplacian-eigenmode validation cells used to show that tMAP responds to controlled ruggedness and graph size.

**Manuscript/SI linkage:** Main Figure 1 and SI Figures 1-6.

This notebook has been pruned to the cells that feed the submitted paper or supporting information. Exploratory cells, stale outputs, and manuscript-extraneous checks were removed.


## Imports and common utilities

Import ruggedness, graph, statistics, and plotting utilities used by the Figure 1 validation workflow.


In [ ]:
# Paper/SI purpose: Imports and common utilities.
# Retained from original cell 0: Import ruggedness, graph, statistics, and plotting utilities used by the Figure 1 validation workflow.

import fitness_landscape as fl
from fitness_landscape.utils import fasta_to_prot20_sequences
from fitness_landscape.core.landscape import FitnessLandscape
from fitness_landscape.transforms.eigenmode import eigenmode_decomposition
from collections import defaultdict
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import re
import pickle
import json
from tqdm import tqdm
import networkx as nx
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import spearmanr
from scipy.stats import norm
import matplotlib as mpl


## Compute dense NK tMAP sweeps

Generate NK landscapes across N, K, and replicate seeds, then compute tMAP for each landscape.


In [ ]:
# Paper/SI purpose: Compute dense NK tMAP sweeps.
# Retained from original cell 6: Generate NK landscapes across N, K, and replicate seeds, then compute tMAP for each landscape.

# Init dict to store replicates
replicate_dict = {}

# Number of replicates
num_replicates = 10
for replicate in range(num_replicates): 

    # Init dict to store results
    tmap_dict = {}

    # Range of N variables to consider
    n_range = list(range(4, 11))

    # Define K up to N-1 for each N
    for n_param in n_range:
        k_range = list(range(0, n_param))
        
        # Construct NK landscapes and compute diffusion map for each (N, K)
        for k_param in k_range:
            nk = fl.models.nk.create_nk_binary_landscape(N=n_param, K=k_param, seed=replicate)
            tmap_dict[(n_param, k_param)] = fl.analysis.diffusion_scale.compute_ruggedness_diffusion_scale(nk, t_min=1e-10, t_max=1e2, prior='uniform')

    # Store replicate results
    replicate_dict[replicate] = tmap_dict


## Plot tMAP versus NK epistasis

Restructure dense NK results and write the main Figure 1 tMAP-versus-K panel.


In [ ]:
# Paper/SI purpose: Plot tMAP versus NK epistasis.
# Retained from original cell 7: Restructure dense NK results and write the main Figure 1 tMAP-versus-K panel.

# replicate_dict: {rep: {(N, K): {'t_map': ... , ...}, ...}, ...}
# Restructure to: restructured[rep][N] -> list of (K, t_map)

restructured = defaultdict(lambda: defaultdict(list))
all_n_values = set()

for rep, tmap_dict in replicate_dict.items():
    for (n, k), res in tmap_dict.items():
        t = res["t_map"]  # <- new field name
        restructured[rep][n].append((k, t))
        all_n_values.add(n)

sorted_n = sorted(all_n_values)

fig, ax = plt.subplots(figsize=(3, 2.75))
cmap = plt.get_cmap("viridis")
colors = {n: cmap(i / (len(sorted_n) - 1 if len(sorted_n) > 1 else 1))
          for i, n in enumerate(sorted_n)}

for rep_data in restructured.values():
    for n_value, points in rep_data.items():
        points.sort(key=lambda x: x[0])  # sort by K
        k_vals = [k for k, _ in points]
        tmap_vals = [t for _, t in points]

        ax.plot(
            tmap_vals, k_vals,
            color=colors[n_value],
            alpha=0.6,
            marker="o",
            markersize=4,
            linestyle="-"
        )

ax.set_xlabel(r"$t_{MAP}$")
ax.set_ylabel("K")
ax.set_xscale("log")
ax.invert_yaxis()
ax.grid(True, which="both", ls="--", c="0.7")

plt.tight_layout()
plt.savefig("../figures/figure_1/tmap_vs_k.pdf")
plt.show()


## Assemble NK statistics table

Convert dense NK replicate results into a tabular form for correlation and regression tests.


In [ ]:
# Paper/SI purpose: Assemble NK statistics table.
# Retained from original cell 9: Convert dense NK replicate results into a tabular form for correlation and regression tests.

# Convert to to long form dataframe
rows = []
for rep, rep_data in restructured.items():
    for N, points in rep_data.items():
        for K, t in points:
            rows.append({
                "replicate": rep,
                "N": N,
                "K": K,
                "t_map": t,
                "log_t_map": np.log(t)
            })

df = pd.DataFrame(rows)


## Report K-tMAP rank correlation

Compute the Spearman association showing that tMAP decreases as NK epistasis K increases.


In [ ]:
# Paper/SI purpose: Report K-tMAP rank correlation.
# Retained from original cell 10: Compute the Spearman association showing that tMAP decreases as NK epistasis K increases.

rho, p_value = spearmanr(df["K"], df["t_map"])
print(f"Spearman correlation between K and t_map: rho={rho:.4f}, p-value={p_value:.4g}")


## Model K effect controlling for N

Fit a robust linear model for log tMAP against K while controlling for sequence-space size.


In [ ]:
# Paper/SI purpose: Model K effect controlling for N.
# Retained from original cell 11: Fit a robust linear model for log tMAP against K while controlling for sequence-space size.

# OLS on parameter K with constant terms for N.
model_K = smf.ols(
    "log_t_map ~ K + C(N)",
    data=df
).fit(cov_type="HC3")  # robust SEs

print(model_K.summary())


## Compute K-model p-value

Recover the two-sided p-value used for the K effect reported with the dense NK validation.


In [ ]:
# Paper/SI purpose: Compute K-model p-value.
# Retained from original cell 12: Recover the two-sided p-value used for the K effect reported with the dense NK validation.

# Comnpute p-value from z-score
z = -14.412
p = 2 * norm.sf(abs(z))
print(p)


## Model N effect controlling for K

Fit the companion robust model testing the apparent size dependence of tMAP.


In [ ]:
# Paper/SI purpose: Model N effect controlling for K.
# Retained from original cell 13: Fit the companion robust model testing the apparent size dependence of tMAP.

# OLS on parameter N with constant terms for K.
model_N = smf.ols( 
    "log_t_map ~ N + C(K)",
    data=df
).fit(cov_type="HC3")  # robust SEs

print(model_N.summary())


## Compute N-model p-value

Recover the two-sided p-value for the sequence-space size term.


In [ ]:
# Paper/SI purpose: Compute N-model p-value.
# Retained from original cell 14: Recover the two-sided p-value for the sequence-space size term.

# Compute p-value from z-score
z = 11.197
p = 2 * norm.sf(abs(z))
print(p)


## Generate Laplacian eigenmode landscapes

Build landscapes whose fitness signal is a chosen Laplacian eigenvector to validate spectral scaling of tMAP.


In [ ]:
# Paper/SI purpose: Generate Laplacian eigenmode landscapes.
# Retained from original cell 16: Build landscapes whose fitness signal is a chosen Laplacian eigenvector to validate spectral scaling of tMAP.

# Init list to store results
results = []

# Range of N variables to consider
n_range = list(range(4, 11))

for _,n_param in tqdm(enumerate(n_range)):

    # Construct NK landscape with dummy K
    nk = fl.models.nk.create_nk_binary_landscape(N=n_param, K=0, seed=0)

    # Perform eigenmode decomposition 
    eigvals, eigvecs = eigenmode_decomposition(
        nk,
        matrix="laplacian",
    )
    
    # Total number of Laplacian eigenvectors (== number of nodes)
    n_eig = eigvecs.shape[1]  # should equal len(nk)

    # Only test first 25% of eigenmodes (skip k=0, )
    max_k = int(len(nk) * 0.50)

    for k in range(1, max_k + 1):
        # Normalized eigenvector index in (0, 1]
        k_norm = k / (n_eig - 1)

        # Attach the eigenvector as the "fitness" layer/signal
        layer_name = f"laplacian_eigvec_{k}"
        nk.attach(
            name=layer_name,
            values=eigvecs[:, k],
            dtype="numeric",
        )
        nk.view(layer_name)

        # Compute diffusion-scale ruggedness (optionally set t_min/t_max)
        tmap_res = fl.analysis.diffusion_scale.compute_ruggedness_diffusion_scale(
            nk,
            t_min=1e-10,
            t_max=1e2,
        )

        # Store everything you need for analysis
        results.append({
            "N": n_param,
            "num_nodes": len(nk),
            "eig_index": k,
            "eig_index_norm": k_norm,
            "eigval": float(eigvals[k]),
            "t_map": float(tmap_res["t_map"]),
            "t_lo": float(tmap_res["t_lower_confidence_interval"]),
            "t_hi": float(tmap_res["t_upper_confidence_interval"]),
            "logpost_map": float(tmap_res["t_logposterior_map"]),
            "var_approx": float(tmap_res["variance_approximate"]),
            "layer": layer_name,
        })

# Convert to DataFrame for convenience
df_eig = pd.DataFrame(results)


## Plot normalized eigenmode index control

Write the Figure 1/SI panel showing tMAP against normalized Laplacian eigenvector index.


In [ ]:
# Paper/SI purpose: Plot normalized eigenmode index control.
# Retained from original cell 17: Write the Figure 1/SI panel showing tMAP against normalized Laplacian eigenvector index.

df = df_eig.copy()

# Collect N values and color-map them
sorted_n = sorted(df["N"].unique())
fig, ax = plt.subplots(figsize=(3, 2.75))
cmap = plt.get_cmap("viridis")
colors = {n: cmap(i / (len(sorted_n) - 1 if len(sorted_n) > 1 else 1))
          for i, n in enumerate(sorted_n)}

# Plot one line per N
for n in sorted_n:
    sub = df[df["N"] == n].sort_values("eig_index_norm")
    ax.plot(
        sub["t_map"].values,
        sub["eig_index_norm"].values,
        color=colors[n],
        alpha=1.0,
        markersize=0,
        linestyle="-",
    )

ax.set_xlabel(r"$t_{MAP}$")
ax.set_ylabel("Normalized Laplacian Index")
ax.set_xscale("log")
ax.invert_yaxis()
ax.grid(True, which="both", ls="--", c="0.7")

plt.tight_layout()
plt.savefig("../figures/figure_1/tmap_vs_norm_laplacian_idx.pdf")
plt.show()


## Model normalized eigenmode effect

Fit the regression showing normalized spectral position accounts for the apparent N dependence.


In [ ]:
# Paper/SI purpose: Model normalized eigenmode effect.
# Retained from original cell 22: Fit the regression showing normalized spectral position accounts for the apparent N dependence.

# OLS on parameter N with constant terms for K.
df['log_t_map'] = np.log(df['t_map'])
model_j = smf.ols( 
    "log_t_map ~ N + eig_index_norm",
    data=df
).fit(cov_type="HC3")  # robust SEs

print(model_j.summary())
